<a href="https://colab.research.google.com/github/matzz-11/Quantum_Colab.ipynb/blob/main/7_Po%C3%A7o_Quadrado_Duplo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Teoria**

---

## **Potencial, Função de onda e Energia**

Esse potencial é apresentado de forma qualitativa para nos ajudar a desenvolver uma intuição física do que irá vir! Basicamente, temos 3 situações que devemos compreender nesse tópico.

- Distância de separação = 0
- Distância de separação → 0
- Distância de separação → ∞

Estamos interessados em ver como as funções de onda e densidades de probabilidade se comportam conforme essa distância se altera. No primeiro caso, é como se tivessemos um grande poço quadrado finito, "com o dobro de largura", sendo análogo ao estudo anterior, apenas com mais espaço!

No segundo caso, com os poços separados, temos uma "abertura" na região, resultado no **tunelamento** entre poços! Esse é um dos casos mais interessantes, pois a probabilidade de tunelar irá depender da energia da partícula estar condizendo com os níveis quantizados de cada poço!

No terceiro caso, aprendemos o fenômeno da **degenerescência**, onde os poços estão tão distantes que começam a se comportar como "únicos", sem acoplamentos. Isso faz com que ambos convirjam para a **mesma energia**, com as funções de onda organizadas de forma simétrica.


---

## **Aplicações**

Uma das aplicações mais interessantes desse modelo é a simulação de sistemas moleculares, como o íon molecular de hidrogênio, com cada poço sendo a força atrativa de um núcleo atômico. Podemos também isolar os dois primeiros níveis de energia de um poço duplo com uma barreira alta, nos aproximando de um qubit (sistema de dois níveis), ou seja, pode ser aplicado também nos estudos da computação quântica!

---

# **Prática**

---

## **Requisitos do Código**
Primeiramente, temos três bibliotecas para construção desse código, sendo:

- numpy
- matplotlib.pyplot
- ipywidgets
- scipy.linalg

Elas foram utilizadas para simplificação de cálculos matemáticos, construções gráficas e criação de interfaces interativas.

---

## **Explicação do código**

Para visualizarmos o poço duplo, foi construído um gráfico com as seguintes informações:

- Azul para a função de onda do estado estacionário,
- Preto para o contorno do potencial (poço),
- Verde para a visualização dos níveis de energia.

Dessa forma, através dos widgets "Estado (n)", "Largura (a)" "Barreira (b)" e "Profund(V0)", podemos visualizar nos gráficos os formatos de cada objeto.

---

## **Código**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
from scipy.linalg import eigh

def plot_foco_nos_pocos(n, a, b, V0):
    margem = 2.0
    L = a + b/2.0 + margem
    N = 2000
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]

    # Potencial
    V = np.zeros_like(x)
    poco_esq = (x >= -b/2.0 - a) & (x <= -b/2.0)
    poco_dir = (x >= b/2.0) & (x <= b/2.0 + a)
    V[poco_esq] = -V0
    V[poco_dir] = -V0

    # Hamiltoniano
    diag = -2.0 * np.ones(N)
    off_diag = np.ones(N - 1)
    T = -1.0 / (dx**2) * (np.diag(diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1))
    H = T + np.diag(V)

    # Energia e estados
    E, psi = eigh(H)

    idx_ligados = E < 0
    E_lig = E[idx_ligados]
    psi_lig = psi[:, idx_ligados]
    num_est = len(E_lig)

    if num_est == 0:
        print(f"O potencial V0={V0} é muito fraco para suportar estados ligados.")
        return

    n_plot = min(n, num_est)
    aviso = f' (Mostrando n={n_plot} pois {n} excede o limite)' if n > num_est else ''

    E_n = E_lig[n_plot - 1]
    psi_n = psi_lig[:, n_plot - 1]

    # Normalização
    psi_n = psi_n / np.sqrt(np.trapz(psi_n**2, x))
    if psi_n[np.argmax(np.abs(psi_n))] < 0:
        psi_n = -psi_n

    # Construção do Gráfico
    plt.figure(figsize=(14, 8))

    # Desenha o contorno do Poço
    plt.plot(x, V, color='black', linewidth=3.5, label='Potencial $V(x)$')
    plt.fill_between(x, V, 0, color='lightgray', alpha=0.4)

    # Desenha todos os níveis de energia em cinza (como degraus)
    for i, Ek in enumerate(E_lig):
        plt.hlines(Ek, -L, L, color='gray', linestyle='--', alpha=0.5)

    # Destaca o Nível de Energia selecionado
    plt.hlines(E_n, -L, L, color='green', linewidth=2.5)
    plt.text(L - 0.8, E_n + V0*0.02, f'$E_{{{n_plot}}}$', color='green', fontsize=14, fontweight='bold')

    # Ajusta a amplitude da função de onda para ela caber visualmente no poço
    # A escala é dinâmica baseada na profundidade do poço e número de estados
    escala = (V0 / max(4, num_est)) * 0.8 / np.max(np.abs(psi_n))
    psi_plot = E_n + escala * psi_n

    # Plota a Função de Onda "surfando" sobre o seu nível de energia
    plt.plot(x, psi_plot, color='blue', linewidth=2.5, label=rf'Função de Onda $\psi_{{{n_plot}}}(x)$')
    plt.fill_between(x, psi_plot, E_n, where=(psi_plot > E_n), color='blue', alpha=0.15)
    plt.fill_between(x, psi_plot, E_n, where=(psi_plot < E_n), color='cyan', alpha=0.15)

    # Formatação Focada
    plt.title(f'Poço Duplo{aviso}', fontsize=16)
    plt.xlabel('Posição ($x$)', fontsize=14)
    plt.ylabel('Energia', fontsize=14)

    # O "Zoom" que mantém o foco apenas nos poços e arredores
    plt.xlim(-a - b/2.0 - 1.5, a + b/2.0 + 1.5)
    # Limite Y abraça do fundo do poço até um pouco acima de E=0
    plt.ylim(-V0 - (V0*0.05), V0 * 0.2)

    plt.legend(loc='lower center', fontsize=12, ncol=2)
    plt.axhline(0, color='black', linewidth=1)
    plt.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

# Widgets
interact(plot_foco_nos_pocos,
         n=widgets.IntSlider(min=1, max=15, step=1, value=1, description='Estado (n):'),
         a=widgets.FloatSlider(min=0.5, max=4.0, step=0.1, value=1.5, description='Largura (a):'),
         b=widgets.FloatSlider(min=0.1, max=4.0, step=0.1, value=0.5, description='Barreira (b):'),
         V0=widgets.FloatSlider(min=5.0, max=80.0, step=1.0, value=30.0, description='Profund(V0):'))